# 00 · Build the US daily panel from WRDS

The paper uses daily China A-share data with 415 characteristics, which is not public. This notebook builds the
closest US equivalent from CRSP:

| | Paper | Here |
|---|---|---|
| Universe | A-shares, excluding ST and suspended stocks | CRSP common stocks (share codes 10, 11) on NYSE/AMEX/Nasdaq, price ≥ $1 |
| Period | 2010 to 2023, daily | 2010 to 2023, daily |
| Target | VWAP(t+1) to VWAP(t+2) return minus CSI 500 | close(t+1) to close(t+2) return minus CRSP value-weighted market |
| Characteristics | 415 daily | 21 computed daily from CRSP, plus optionally about 200 monthly OSAP signals (used with a one-month lag) |
| Industries | 31 Shenwan dummies | 10 SIC-division dummies |

Each characteristic is ranked across stocks within each day to [-1, 1], with missing values set to 0, and stored as int8.

**Before running:** `pip install wrds` (and `pip install openassetpricing` if `USE_OSAP = True`). The first
`wrds.Connection` asks for your WRDS password and offers to save it in a pgpass file. Downloading 2007 to 2024
takes a while; files already in `data/raw` are skipped. Building the features needs about 8 GB of RAM.

In [ ]:
from pathlib import Path

import pandas as pd

from mlm import crsp
from mlm.data import load_panel, panel_features

WRDS_USERNAME = "your_wrds_username"
RAW_DIR, PANEL_DIR = Path("data/raw"), Path("data/panel")
DOWNLOAD_YEARS = (2007, 2024)   # 3 years of history before 2010 for mom36_12; 2024 for the t+2 returns at the end of 2023
PANEL_START, PANEL_END = "2010-01-01", "2023-12-31"
MIN_PRICE = 1.0
EXEC_LAG = 1                    # trade at the close of t+1, earn the t+2 return
USE_OSAP = False                # start with the 21 daily characteristics; True adds the OSAP monthly signals

In [ ]:
import wrds

db = wrds.Connection(wrds_username=WRDS_USERNAME)
crsp.download(db, RAW_DIR, *DOWNLOAD_YEARS)
db.close()

In [ ]:
dsf, mkt = crsp.load_raw(RAW_DIR)
panel, numeric, dummies = crsp.build_features(dsf, mkt, PANEL_START, PANEL_END, MIN_PRICE, EXEC_LAG)
del dsf
osap = crsp.load_osap(RAW_DIR) if USE_OSAP else None
features = crsp.write_ranked_panel(panel, PANEL_DIR, numeric, dummies, osap,
                                   min_price=MIN_PRICE, exec_lag=EXEC_LAG, osap=USE_OSAP)
print(len(features), "features")

## Sanity checks: stocks per day by year (compare with Table A.1) and target distribution

In [ ]:
counts = panel.groupby("date").size()
pd.DataFrame({"Avg stocks per day": counts.groupby(counts.index.year).mean().round(0),
              "Days": counts.groupby(counts.index.year).size()})

In [ ]:
panel["target"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999])